# DiffAM - Final Notebook
**Before running:** `Runtime → Change runtime type → T4 GPU → Save`

Run cells top to bottom. Each cell is independent and fast.

In [1]:
# CELL 1: Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'NO GPU - go to Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4


In [2]:
# CELL 2: Install deps + clone repo
!apt-get install -y cmake libopenblas-dev liblapack-dev -q
!git clone https://github.com/HansSunY/DiffAM.git /content/DiffAM 2>/dev/null || echo 'already cloned'
!sed -i 's/scipy==1.10.1/scipy==1.11.4/' /content/DiffAM/requirements.txt
!pip install -r /content/DiffAM/requirements.txt -q
!pip install git+https://github.com/openai/CLIP.git -q
print('DONE')

Reading package lists...
Building dependency tree...
Reading state information...
liblapack-dev is already the newest version (3.10.0-2ubuntu1).
libopenblas-dev is already the newest version (0.3.20+ds-1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
already cloned
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for dlib
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (dlib)
  Preparing metadata (setup.py) ... done
DONE


In [3]:
# CELL 3: Verify imports
import torch, scipy, clip, dlib
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
print('scipy:', scipy.__version__, '| dlib: OK | clip: OK')

torch: 2.10.0+cu128 | cuda: True
scipy: 1.16.3 | dlib: OK | clip: OK


In [4]:
# CELL 4: Symlink pretrained weights (already on Drive from before)
import os, shutil

# The weights are at one of these two locations depending on previous run
# Check which one has the files
nested = '/content/drive/MyDrive/DiffAM_project/pretrained/pretrained'
flat   = '/content/drive/MyDrive/DiffAM_project/pretrained'

if os.path.exists(os.path.join(nested, 'celeba_hq.ckpt')):
    weights_path = nested
elif os.path.exists(os.path.join(flat, 'celeba_hq.ckpt')):
    weights_path = flat
else:
    raise FileNotFoundError('Pretrained weights not found on Drive! Re-download them.')

# Symlink into repo
target = '/content/DiffAM/pretrained'
if os.path.islink(target) or os.path.exists(target):
    os.system(f'rm -rf {target}')
os.symlink(weights_path, target)

print('Weights found at:', weights_path)
print('Files:', os.listdir(target))

Weights found at: /content/drive/MyDrive/DiffAM_project/pretrained
Files: ['makeup.pt', 'shape_predictor_68_face_landmarks.dat', 'model_ir_se50.pth', 'celeba_hq.ckpt']


In [5]:
# CELL 5: Download assets.zip to LOCAL disk and unzip (fast - not through Drive)
import os

!pip install gdown -q

# Download directly to local fast SSD
!gdown "1IKiWLv99eUbv3llpj-dOegF3O7FWW29J" -O /content/assets.zip

# Unzip locally
!unzip -o -q /content/assets.zip -d /content/assets_extracted/
!rm /content/assets.zip

print('Extracted contents:')
!ls /content/assets_extracted/

Downloading...
From (original): https://drive.google.com/uc?id=1IKiWLv99eUbv3llpj-dOegF3O7FWW29J
From (redirected): https://drive.google.com/uc?id=1IKiWLv99eUbv3llpj-dOegF3O7FWW29J&confirm=t&uuid=18fc470c-aa20-4ade-b65a-bd17748f4ba9
To: /content/assets.zip
100% 1.20G/1.20G [00:23<00:00, 51.0MB/s]
Extracted contents:
assets


In [6]:
# CELL 6: Put assets in the right place
import os, shutil

extracted = '/content/assets_extracted'
contents = os.listdir(extracted)
print('Extracted contents:', contents)

# Handle if zip extracted into a subfolder called 'assets'
if contents == ['assets']:
    src = os.path.join(extracted, 'assets')
else:
    src = extracted

# Move to /content/assets/
if os.path.exists('/content/assets'):
    shutil.rmtree('/content/assets')
shutil.move(src, '/content/assets')

# Make sure datasets folder exists inside
os.makedirs('/content/assets/datasets', exist_ok=True)

# Symlink into repo
repo_assets = '/content/DiffAM/assets'
if os.path.islink(repo_assets) or os.path.exists(repo_assets):
    os.system(f'rm -rf {repo_assets}')
os.symlink('/content/assets', repo_assets)

print('assets contents:', os.listdir('/content/assets'))
print('datasets contents:', os.listdir('/content/assets/datasets'))

Extracted contents: ['assets']
assets contents: ['datasets', 'models']
datasets contents: ['MT-dataset', 'target', 'test', 'pairs']


In [7]:
# CELL 7: Unzip CelebAMask-HQ to LOCAL disk (fast)
# The zip is already on your Drive from the server-side copy earlier
import os

celeb_zip = '/content/drive/MyDrive/DiffAM_project/CelebAMask-HQ.zip'

if not os.path.exists(celeb_zip):
    print('ERROR: CelebAMask-HQ.zip not on Drive!')
    print('It should be at:', celeb_zip)
    print('Check your Drive manually.')
else:
    print(f'Found zip: {os.path.getsize(celeb_zip)/1e9:.1f} GB')
    print('Unzipping to local disk (fast)...')
    !unzip -o -q "{celeb_zip}" -d /content/
    print('Done! Contents:')
    !ls /content/ | grep -i celeb

Found zip: 3.2 GB
Unzipping to local disk (fast)...
Done! Contents:
CelebAMask-HQ


In [8]:
# CELL 8: Symlink CelebAMask-HQ into datasets
import os

# Find where it unzipped to
candidates = ['/content/CelebAMask-HQ', '/content/CelebAMask-HQ-dataset']
celeb_local = None
for c in candidates:
    if os.path.exists(c):
        celeb_local = c
        break

if celeb_local is None:
    # Check what's in /content
    print('Could not find CelebAMask-HQ folder. Contents of /content:')
    !ls /content/
else:
    target = '/content/assets/datasets/CelebAMask-HQ'
    if os.path.islink(target) or os.path.exists(target):
        os.system(f'rm -rf {target}')
    os.symlink(celeb_local, target)
    print('Symlinked', celeb_local, '->', target)
    print('Sample contents:', os.listdir(target)[:5])

Symlinked /content/CelebAMask-HQ -> /content/assets/datasets/CelebAMask-HQ
Sample contents: ['CelebA-HQ-img', 'CelebAMask-HQ-mask-anno', 'CelebA-HQ-to-CelebA-mapping.txt', '.DS_Store', 'CelebAMask-HQ-pose-anno.txt']


In [9]:
# CELL 9: FINAL STRUCTURE CHECK
print('=== pretrained ===')
!ls /content/DiffAM/pretrained/

print('\n=== assets ===')
!ls /content/DiffAM/assets/

print('\n=== datasets ===')
!ls /content/DiffAM/assets/datasets/

print('\n=== models ===')
!ls /content/DiffAM/assets/models/

=== pretrained ===
celeba_hq.ckpt	model_ir_se50.pth
makeup.pt	shape_predictor_68_face_landmarks.dat

=== assets ===
datasets  models

=== datasets ===
CelebAMask-HQ  MT-dataset  pairs  target  test

=== models ===
facenet.pth  ir152.pth	irse50.pth  mobile_face.pth
facenet.py   ir152.py	irse.py     __pycache__


In [12]:
%%bash
# Uninstall the conflicting HuggingFace datasets package
pip uninstall datasets -y

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0


In [14]:
%%bash
pip install lpips -q

In [18]:
%%bash
grep "MT_adv_loss_w" /content/DiffAM/main.py

    parser.add_argument('--MT_adv_loss_w', type=int, default=0.5, help='Weights of adv loss')


In [19]:
%%bash
sed -i "s/MT_adv_loss_w.*type=int/MT_adv_loss_w', type=float/" /content/DiffAM/main.py

In [28]:
with open('/content/DiffAM/models/ddpm/diffusion.py', 'r') as f:
    code = f.read()

# 1. Add import at top
old_import = 'import math\nimport torch\nimport torch.nn as nn'
new_import = 'import math\nimport torch\nimport torch.nn as nn\nfrom torch.utils.checkpoint import checkpoint'

# 2. Patch down-blocks (line 310)
old_down = '                h = self.down[i_level].block[i_block](hs[-1], temb)'
new_down = '                h = checkpoint(self.down[i_level].block[i_block], hs[-1], temb, use_reentrant=False)'

# 3. Patch middle blocks (lines 319, 321)
old_mid1 = '        h = self.mid.block_1(h, temb)'
new_mid1 = '        h = checkpoint(self.mid.block_1, h, temb, use_reentrant=False)'

old_mid2 = '        h = self.mid.block_2(h, temb)'
new_mid2 = '        h = checkpoint(self.mid.block_2, h, temb, use_reentrant=False)'

# 4. Patch up-blocks (lines 326-327)
old_up = ('                h = self.up[i_level].block[i_block](\n'
          '                    torch.cat([h, hs.pop()], dim=1), temb)')
new_up  = ('                h = checkpoint(self.up[i_level].block[i_block],\n'
           '                    torch.cat([h, hs.pop()], dim=1), temb, use_reentrant=False)')

# Apply all patches with verification
patches = [
    (old_import, new_import, 'import'),
    (old_down,   new_down,   'down-block'),
    (old_mid1,   new_mid1,   'mid-block-1'),
    (old_mid2,   new_mid2,   'mid-block-2'),
    (old_up,     new_up,     'up-block'),
]

for old, new, name in patches:
    if old in code:
        code = code.replace(old, new)
        print(f'OK: {name} patched')
    else:
        print(f'FAILED: {name} - exact string not found!')

# Only write if ALL patches succeeded
if all(old in open('/content/DiffAM/models/ddpm/diffusion.py').read() or new in code
       for old, new, _ in patches):
    with open('/content/DiffAM/models/ddpm/diffusion.py', 'w') as f:
        f.write(code)
    print('\nFile written. Verifying key lines:')
    for i, line in enumerate(code.split('\n'), 1):
        if 'checkpoint' in line:
            print(f'  {i}: {line}')

OK: import patched
OK: down-block patched
OK: mid-block-1 patched
OK: mid-block-2 patched
OK: up-block patched

File written. Verifying key lines:
  4: from torch.utils.checkpoint import checkpoint
  311:                 h = checkpoint(self.down[i_level].block[i_block], hs[-1], temb, use_reentrant=False)
  320:         h = checkpoint(self.mid.block_1, h, temb, use_reentrant=False)
  322:         h = checkpoint(self.mid.block_2, h, temb, use_reentrant=False)
  327:                 h = checkpoint(self.up[i_level].block[i_block],


In [29]:
# CELL 10: RUN MAKEUP TRANSFER (paper reproduction config)
import os, torch
os.chdir('/content/DiffAM')

# Free all unused GPU memory before starting
torch.cuda.empty_cache()

!PYTHONPATH=/content/DiffAM python main.py \
  --makeup_transfer \
  --config celeba.yml \
  --exp /content/drive/MyDrive/DiffAM_project/runs/test \
  --do_train 1 --do_test 1 \
  --n_train_img 200 --n_test_img 100 \
  --n_iter 6 --t_0 60 \
  --n_inv_step 20 --n_train_step 6 --n_test_step 6 \
  --lr_clip_finetune 4e-6 \
  --MT_iter_without_adv 2 \
  --MT_adv_loss_w 1.2 \
  --model_path pretrained/celeba_hq.ckpt \
  --target_img 3 \
  --target_model 3 \
  --ref_img 'XMY-060'

INFO - main.py - 2026-03-01 13:02:14,800 - Using device: cuda
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
INFO - main.py - 2026-03-01 13:02:14,804 - Exp instance id = 22948
INFO - main.py - 2026-03-01 13:02:14,804 - Exp comment = 
INFO - main.py - 2026-03-01 13:02:14,804 - Config =
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
/content/DiffAM/utils/image_processing.py:142: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
/content/drive/MyDrive/DiffAM_project/runs/test_MT_CelebA_HQ_XMY-060_t60_ninv20_ngen6_dis1_dir0.3_lr4e-06
Transfer makeup style XMY-060
Original diffusion Model loaded.
Setting optimizer with lr=4

In [27]:
# Show me the exact lines around the up-block forward pass
with open('/content/DiffAM/models/ddpm/diffusion.py', 'r') as f:
    lines = f.readlines()

# Print full file with line numbers
for i, line in enumerate(lines, start=1):
    print(f"{i:4d}: {line}", end='')

   1: import math
   2: import torch
   3: import torch.nn as nn
   4: 
   5: 
   6: def get_timestep_embedding(timesteps, embedding_dim):
   7:     """
   8:     This matches the implementation in Denoising Diffusion Probabilistic Models:
   9:     From Fairseq.
  10:     Build sinusoidal embeddings.
  11:     This matches the implementation in tensor2tensor, but differs slightly
  12:     from the description in Section 3.5 of "Attention Is All You Need".
  13:     """
  14:     assert len(timesteps.shape) == 1
  15: 
  16:     half_dim = embedding_dim // 2
  17:     emb = math.log(10000) / (half_dim - 1)
  18:     emb = torch.exp(torch.arange(half_dim, dtype=torch.float32) * -emb)
  19:     emb = emb.to(device=timesteps.device)
  20:     emb = timesteps.float()[:, None] * emb[None, :]
  21:     emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
  22:     if embedding_dim % 2 == 1:  # zero pad
  23:         emb = torch.nn.functional.pad(emb, (0, 1, 0, 0))
  24:     return emb
 

In [22]:
%%bash
# Fix 1: free cache before CLIP loss computation
sed -i 's/loss_dir = (2 - clip_loss_func(x0/torch.cuda.empty_cache()\n                        loss_dir = (2 - clip_loss_func(x0/' /content/DiffAM/makeup_transfer.py

#